# DirectQuery Semantic Model - Best Practices Check

This notebook inspects a Power BI / Fabric semantic model that uses **DirectQuery** and validates it against the best-practice checklist:

1. Star schema fundamentals (fact/dimension shape)
1b. Power Query transformations on DQ fact tables -> recommend a **view at the data source**
2. All Dimension tables in **Dual** storage mode
3. **Assume Referential Integrity** enabled on relationships
4. Relationship columns are **Integer** data type
5. Aggregation strategy: prefer targeted **User-Defined Aggregations**; inspect Auto Aggregations and SSO limitations
6. Query parallelism / `MaxParallelismPerQuery` configured
7. SKU-appropriate DQ concurrent connections / parallelism

Additional advisory optimizations cover **calendar-based time intelligence**, **Visual Calculations**, **Hybrid Tables** with `dataCoverageDefinition`, and **Discourage Implicit Measures**.

> Run this notebook inside a **Microsoft Fabric** workspace (the `sempy` / Semantic Link library is pre-installed there). It can also be run locally if you install `semantic-link` and authenticate.

References: [DirectQuery guidance](https://learn.microsoft.com/power-bi/guidance/directquery-model-guidance), [Storage modes](https://learn.microsoft.com/power-bi/transform-model/desktop-storage-mode), [Assume RI](https://learn.microsoft.com/power-bi/connect-data/desktop-assume-referential-integrity), [User-defined aggregations](https://learn.microsoft.com/power-bi/transform-model/aggregations-advanced).

## 1. Install & import Semantic Link
`sempy` is already available in Fabric notebooks. Uncomment the `%pip install` line if running elsewhere.

In [2]:
%pip install semantic-link --quiet

import sempy.fabric as fabric
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

^C
Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'sempy'

## 2. Configure the target semantic model
Set `WORKSPACE` and `DATASET` to the workspace + semantic model you want to audit. Leave `WORKSPACE = None` to use the current workspace.

In [ ]:
WORKSPACE = "PBICopilot"             # e.g. "Sales Analytics" or a workspace GUID
DATASET   = "DirectQueryUDAggregation"

# List available datasets in the workspace for convenience
fabric.list_datasets(workspace=WORKSPACE)

## 3. Pull model metadata
Retrieve tables, columns, relationships and partitions from the semantic model.

In [ ]:
tables        = fabric.list_tables(DATASET, workspace=WORKSPACE, additional_xmla_properties=["Description"])
columns       = fabric.list_columns(DATASET, workspace=WORKSPACE, additional_xmla_properties=["SourceColumn"])
relationships = fabric.list_relationships(DATASET, workspace=WORKSPACE)
partitions    = fabric.list_partitions(DATASET, workspace=WORKSPACE)

print(f"Tables:        {len(tables)}")
print(f"Columns:       {len(columns)}")
print(f"Relationships: {len(relationships)}")
print(f"Partitions:    {len(partitions)}")
partitions.head()

## 4. Identify DirectQuery usage & storage modes
Each partition exposes a `Mode` (`Import`, `DirectQuery`, `Dual`). A table is considered DirectQuery if any of its partitions use DQ.

In [ ]:
mode_col = "Mode" if "Mode" in partitions.columns else "Storage Mode"
table_col = "Table Name" if "Table Name" in partitions.columns else "TableName"

storage = (partitions.groupby(table_col)[mode_col]
                     .agg(lambda s: ",".join(sorted(set(s))))
                     .reset_index()
                     .rename(columns={table_col: "Table", mode_col: "StorageMode"}))

is_dq_model = storage["StorageMode"].str.contains("DirectQuery").any()
print("Model uses DirectQuery:", is_dq_model)
storage

## 5. Classify fact vs dimension tables
Heuristic: a **fact** table participates on the *many* side of one or more relationships; a **dimension** is on the *one* side. This lets us evaluate star-schema shape and required storage modes.

In [ ]:
from_col = "From Table"
to_col   = "To Table"

fact_tables      = set(relationships[from_col].unique())   # many side
dimension_tables = set(relationships[to_col].unique())     # one side
# A table that is only on the one-side is a pure dimension
pure_dimensions  = dimension_tables - fact_tables
pure_facts       = fact_tables - dimension_tables

classification = pd.DataFrame({
    "Table": sorted(set(storage["Table"])),
})
classification["Role"] = classification["Table"].apply(
    lambda t: "Fact" if t in pure_facts else ("Dimension" if t in pure_dimensions else ("Bridge/Other" if t in fact_tables else "Standalone"))
)
classification = classification.merge(storage, on="Table", how="left")
classification

## Check 1 — Star schema fundamentals
Flag snowflaking (dimensions joined to other dimensions) and standalone tables.

In [ ]:
# Snowflake: a table on the 'many' side that is itself a dimension of another table
snowflake_rels = relationships[
    relationships[from_col].isin(pure_dimensions | dimension_tables) &
    ~relationships[from_col].isin(pure_facts)
]

standalone = classification[classification["Role"] == "Standalone"]["Table"].tolist()

print("Snowflaked relationships (dimension -> dimension):", len(snowflake_rels))
display(snowflake_rels)
print("Standalone tables (no relationships):", standalone)

## Check 1b — Power Query transformations on DirectQuery fact tables

In DirectQuery, Power Query (M) steps are folded into the SQL sent to the source. Non-trivial transformations on a **DirectQuery fact table** (filters, added/derived columns, merges, type changes, grouping) add complexity and can break query folding, hurting performance.

**Best practice:** move that logic into a **view at the data source** and point the DirectQuery fact table at the view. This cell reads each DQ table's M expression via TOM, detects transformation steps, and flags fact tables that should be backed by a source view.

In [ ]:
import re

# M functions / step names that indicate a transformation beyond simple source navigation.
TRANSFORM_PATTERNS = [
    r"Table\.TransformColumns", r"Table\.AddColumn", r"Table\.SelectRows",
    r"Table\.RemoveColumns", r"Table\.RenameColumns", r"Table\.TransformColumnTypes",
    r"Table\.Group", r"Table\.Pivot", r"Table\.Unpivot", r"Table\.Distinct",
    r"Table\.Sort", r"Table\.Join", r"Table\.NestedJoin", r"Table\.FillDown",
    r"Table\.FillUp", r"Table\.SplitColumn", r"Table\.CombineColumns",
    r"Table\.ReplaceValue", r"Table\.Buffer", r"Table\.ExpandTableColumn",
    r"Table\.ExpandRecordColumn", r"Table\.AddIndexColumn",
    r"#\"Filtered Rows\"", r"#\"Added Custom\"", r"#\"Grouped Rows\"",
    r"#\"Removed Columns\"", r"#\"Changed Type\"", r"#\"Merged Queries\"",
    r"#\"Added Conditional Column\"", r"#\"Changed Type with Locale\"",
    r"#\"Added Custom Column\"", r"#\"Inserted ",
]
_transform_re = re.compile("|".join(TRANSFORM_PATTERNS))

def detect_transformations(m_expr: str):
    if not m_expr:
        return []
    return sorted({m.group(0) for m in _transform_re.finditer(m_expr)})

# DirectQuery tables from the storage summary (Check 4).
dq_tables = set(storage.loc[storage["StorageMode"].str.contains("DirectQuery"), "Table"])

pq_rows = []
try:
    # Reuse `model` from an earlier cell if present, otherwise open TOM.
    if "model" not in dir() or model is None:
        tom = fabric.create_tom_server(workspace=WORKSPACE)
        db = (tom.Databases.GetByName(DATASET)
              if hasattr(tom.Databases, "GetByName") else tom.Databases[DATASET])
        model = db.Model

    for t in model.Tables:
        if t.Name not in dq_tables:
            continue
        exprs = []
        for p in t.Partitions:
            src = getattr(p, "Source", None)
            expr = getattr(src, "Expression", None)
            if expr:
                exprs.append(str(expr))
        m_all = "\n".join(exprs)
        matches = detect_transformations(m_all)
        role = ("Fact" if t.Name in pure_facts
                else ("Dimension" if t.Name in pure_dimensions else "Bridge/Other"))
        pq_rows.append({
            "Table": t.Name,
            "Role": role,
            "HasMExpression": bool(m_all),
            "HasTransformations": bool(matches),
            "Transformations": ", ".join(matches) if matches else "(none — simple source navigation)",
        })
except Exception as ex:
    print("TOM read failed:", ex)

pq_df = pd.DataFrame(pq_rows)

fact_dq_transforms = (pq_df[(pq_df["Role"] == "Fact") & (pq_df["HasTransformations"])]["Table"].tolist()
                      if not pq_df.empty else [])
other_dq_transforms = (pq_df[(pq_df["Role"] != "Fact") & (pq_df["HasTransformations"])]["Table"].tolist()
                       if not pq_df.empty else [])

VIEW_DOC = "https://learn.microsoft.com/power-bi/guidance/directquery-model-guidance"
if fact_dq_transforms:
    pq_transform_status = "WARN"
    pq_transform_detail = ("Create a VIEW at the data source for DQ fact table(s) with M transformations: "
                           f"{', '.join(fact_dq_transforms)}. See {VIEW_DOC}")
    print("⚠️  DirectQuery FACT tables with Power Query transformations — create a source-level VIEW:")
    for t in fact_dq_transforms:
        print(f"   • {t}")
elif other_dq_transforms:
    pq_transform_status = "WARN"
    pq_transform_detail = ("DQ (non-fact) table(s) use M transformations — consider a source view: "
                           f"{', '.join(other_dq_transforms)}")
    print("⚠️  DirectQuery tables with transformations (non-fact):", ", ".join(other_dq_transforms))
else:
    pq_transform_status = "PASS"
    pq_transform_detail = "No non-trivial Power Query transformations on DirectQuery tables."
    print("✅ No non-trivial Power Query transformations on DirectQuery tables.")

pq_df if not pq_df.empty else pd.DataFrame({"info": ["No DirectQuery tables with M partitions found"]})

## Check 2 — All dimension tables must be in **Dual** mode
When the fact is DirectQuery, dimensions should be Dual so slicers / filter cardinality can resolve locally.

In [ ]:
dim_modes = classification[classification["Role"] == "Dimension"].copy()
dim_modes["DualMode_OK"] = dim_modes["StorageMode"].eq("Dual")

not_dual = dim_modes[~dim_modes["DualMode_OK"]]
print(f"Dimensions NOT in Dual mode: {len(not_dual)}")
not_dual

## Check 3 — Assume Referential Integrity on every relationship
The `RelyOnReferentialIntegrity` flag (a.k.a. Assume RI) enables inner joins on the source and is critical for DQ performance.

In [ ]:
ri_col = next((c for c in relationships.columns if c.lower().replace(" ", "") in
               ("relyonreferentialintegrity", "assumereferentialintegrity")), None)

if ri_col is None:
    # Fetch the property explicitly
    relationships = fabric.list_relationships(
        DATASET, workspace=WORKSPACE,
        additional_xmla_properties=["RelyOnReferentialIntegrity"])
    ri_col = "RelyOnReferentialIntegrity"

missing_ri = relationships[relationships[ri_col].astype(str).str.lower().isin(["false", "0", "nan"])]
print(f"Relationships missing Assume RI: {len(missing_ri)} / {len(relationships)}")
missing_ri[[from_col, "From Column", to_col, "To Column", ri_col]]

## Check 4 — Relationship columns must be Integer
String / GUID / double-typed join keys kill DQ performance.

In [ ]:
cols = columns.rename(columns={"Table Name": "Table", "Column Name": "Column"})

def lookup_type(table, column):
    m = cols[(cols["Table"] == table) & (cols["Column"] == column)]
    return m.iloc[0]["Data Type"] if not m.empty else None

rel_types = relationships.copy()
rel_types["FromType"] = rel_types.apply(lambda r: lookup_type(r[from_col], r["From Column"]), axis=1)
rel_types["ToType"]   = rel_types.apply(lambda r: lookup_type(r[to_col],   r["To Column"]),   axis=1)

INT_TYPES = {"Int64", "Integer", "Whole Number", "Int32"}
bad_types = rel_types[~(rel_types["FromType"].isin(INT_TYPES) & rel_types["ToType"].isin(INT_TYPES))]
print(f"Relationships with non-integer keys: {len(bad_types)}")
bad_types[[from_col, "From Column", "FromType", to_col, "To Column", "ToType"]]

## Check 5 - Aggregation strategy

For most DirectQuery models, prefer targeted **[User-Defined Aggregations](https://learn.microsoft.com/power-bi/transform-model/aggregations-advanced)** designed from known report workloads. They are easier to reason about and control than Auto Aggregations, which can consume substantial capacity units (CUs).

Auto-Aggregation training is a **service-side dataset setting** and is *not* a direct property on the TOM `Model`. This informational check detects it two ways:

1. **TOM side** - look for aggregation tables (hidden tables with columns that have `AlternateOf` set). When auto-agg has trained at least once, these objects exist.
2. **Service side** - call the Power BI REST API (`/datasets/{id}/queryScaleOutSettings` and the dataset settings endpoint) via `sempy`'s `PowerBIRestClient` to read the actual training toggle.

If Auto Aggregations are enabled, review their CU consumption against measured query benefit. Their absence is not a warning. The next check handles SSO separately because Auto Aggregations cannot train over SSO-enabled DirectQuery sources.

In [ ]:
# ---------------------------------------------------------------
# (A) TOM-side detection: aggregation and advanced-model metadata
# ---------------------------------------------------------------
auto_agg_tom = False
agg_columns = []
max_parallel = None
default_mode = None
discourage_implicit_measures = None
hybrid_tables = []
data_coverage_tables = []

try:
    tom = fabric.create_tom_server(workspace=WORKSPACE)
    db = (tom.Databases.GetByName(DATASET)
          if hasattr(tom.Databases, "GetByName") else tom.Databases[DATASET])
    model = db.Model

    max_parallel = getattr(model, "MaxParallelismPerQuery", None)
    default_mode = getattr(model, "DefaultMode", None)
    discourage_implicit_measures = getattr(model, "DiscourageImplicitMeasures", None)

    for table in model.Tables:
        # Auto-agg training creates aggregation columns bound via AlternateOf.
        for column in table.Columns:
            alternate = getattr(column, "AlternateOf", None)
            if alternate is not None:
                agg_columns.append((table.Name, column.Name, str(getattr(alternate, "Summarization", "")),
                                    str(getattr(alternate, "BaseColumn", "") or getattr(alternate, "BaseTable", ""))))

        partition_modes = {str(getattr(partition, "Mode", "")).lower() for partition in table.Partitions}
        if any("import" in mode for mode in partition_modes) and any("directquery" in mode for mode in partition_modes):
            hybrid_tables.append(table.Name)

        for partition in table.Partitions:
            if getattr(partition, "DataCoverageDefinition", None) is not None:
                data_coverage_tables.append(table.Name)
                break

    auto_agg_tom = len(agg_columns) > 0

    print("DefaultMode:                 ", default_mode)
    print("MaxParallelismPerQuery:      ", max_parallel)
    print("DiscourageImplicitMeasures:  ", discourage_implicit_measures)
    print(f"Aggregation columns found:   {len(agg_columns)}")
    print("Hybrid tables:               ", hybrid_tables or "none detected")
    print("Tables with data coverage:   ", data_coverage_tables or "none detected")
    if agg_columns:
        display(pd.DataFrame(agg_columns, columns=["Table", "Column", "Summarization", "BaseObject"]))
except Exception as ex:
    print("TOM access not available:", ex)

# ---------------------------------------------------------------
# (B) Service-side detection: Power BI REST API
# ---------------------------------------------------------------
auto_agg_service = None
try:
    client = fabric.PowerBIRestClient()
    ds_id = fabric.resolve_dataset_id(DATASET, workspace=WORKSPACE)
    ws_id = fabric.resolve_workspace_id(WORKSPACE) if WORKSPACE else None

    url = (f"v1.0/myorg/groups/{ws_id}/datasets/{ds_id}"
           if ws_id else f"v1.0/myorg/datasets/{ds_id}")
    resp = client.get(url).json()

    for key in ("isAutomaticAggregationsEnabled", "autoAggregationsEnabled"):
        if key in resp:
            auto_agg_service = resp[key]
            print(f"REST property `{key}`:", auto_agg_service)
            break
    else:
        print("Auto-aggregation property not present in REST response. Full settings keys:",
              list(resp.keys()))
except Exception as ex:
    print("REST API check skipped:", ex)

# ---------------------------------------------------------------
# Informational verdict: absence is not a warning
# ---------------------------------------------------------------
auto_agg_enabled = bool(auto_agg_tom) or bool(auto_agg_service)
print("\nAuto Aggregations enabled:", auto_agg_enabled,
      f"(tom={auto_agg_tom}, service={auto_agg_service})")

## Check 5b — Single Sign-On (SSO) vs Auto Aggregations

If a DirectQuery data source is configured for **Single Sign-On (SSO)**, Auto Aggregations training **skips** those tables — it logs a warning in the refresh history and won't build system aggregations for the SSO source, because it can't honor per-user, source-level security ([Automatic aggregations — considerations #9](https://learn.microsoft.com/fabric/enterprise/powerbi/aggregations-auto#considerations-and-limitations)). Import-mode aggregation tables over SSO sources are also ignored (Aug 2022+) for security reasons.

**Recommendation when SSO is on:** build **[User-Defined Aggregations](https://learn.microsoft.com/power-bi/transform-model/aggregations-advanced)** instead of relying on Auto Aggregations.

SSO is set on the **connection / gateway data source binding**, not in the model (TOM), so this check tries the Power BI REST API and falls back to a **MANUAL** check when it can't be determined automatically.

In [ ]:
# ---------------------------------------------------------------
# Detect SSO on the model's DirectQuery data source(s) via REST.
# SSO lives on the gateway / connection binding, not in TOM.
# ---------------------------------------------------------------
sso_enabled = None          # True / False / None (unknown)
sso_sources = []

def _is_sso(value) -> bool:
    # singleSignOnType is "None" when off; anything else means some SSO mode.
    return bool(value) and str(value).strip().lower() not in ("none", "0", "false", "")

try:
    client = fabric.PowerBIRestClient()
    ds_id = fabric.resolve_dataset_id(DATASET, workspace=WORKSPACE)
    ws_id = fabric.resolve_workspace_id(WORKSPACE) if WORKSPACE else None

    ds_url = (f"v1.0/myorg/groups/{ws_id}/datasets/{ds_id}/datasources"
              if ws_id else f"v1.0/myorg/datasets/{ds_id}/datasources")
    datasources = client.get(ds_url).json().get("value", [])
    print(f"Bound data sources: {len(datasources)}")

    for datasource in datasources:
        gateway_id = datasource.get("gatewayId")
        datasource_id = datasource.get("datasourceId")
        datasource_type = datasource.get("datasourceType")
        sso_type = None

        # Gateway-bound sources expose singleSignOnType on the gateway datasource.
        if gateway_id and datasource_id:
            try:
                gateway_source = client.get(
                    f"v1.0/myorg/gateways/{gateway_id}/datasources/{datasource_id}"
                ).json()
                sso_type = gateway_source.get("singleSignOnType") or (
                    gateway_source.get("credentialDetails", {}) or {}
                ).get("useEndUserOAuth2Credentials")
            except Exception as gateway_ex:
                print(f"  Gateway datasource read skipped ({datasource_id}):", gateway_ex)

        this_sso = _is_sso(sso_type)
        sso_sources.append({
            "DatasourceType": datasource_type,
            "GatewayId": gateway_id,
            "SingleSignOnType": sso_type,
            "SSO": this_sso,
        })

    if sso_sources:
        sso_enabled = any(source["SSO"] for source in sso_sources)
    display(pd.DataFrame(sso_sources) if sso_sources else pd.DataFrame({"info": ["No bound data sources returned"]}))

except Exception as ex:
    print("SSO REST detection unavailable:", ex)

# ---------------------------------------------------------------
# Verdict
# ---------------------------------------------------------------
UDA_DOC = "https://learn.microsoft.com/power-bi/transform-model/aggregations-advanced"

if sso_enabled is True:
    sso_status = "WARN"
    sso_detail = ("DirectQuery source uses SSO - Auto Aggregations training will skip these tables. "
                  f"Create User-Defined Aggregations instead: {UDA_DOC}")
    print("\nSSO detected. Auto Aggregations will not cover SSO tables.")
    print(f"Recommend User-Defined Aggregations: {UDA_DOC}")
elif sso_enabled is False:
    sso_status = "PASS"
    sso_detail = ("No SSO detected. Prefer targeted User-Defined Aggregations when aggregations are needed; "
                  "treat Auto Aggregations as optional and validate their CU cost.")
    print("\nNo SSO detected on the model's data sources.")
else:
    sso_status = "MANUAL"
    sso_detail = ("Could not determine SSO automatically. Check the connection in 'Manage connections and "
                  "gateways' / semantic model settings. If SSO is ON, use User-Defined Aggregations: " + UDA_DOC)
    print("\nMANUAL CHECK: verify SSO on the data source connection.")
    print(f"If SSO is ON, use User-Defined Aggregations: {UDA_DOC}")

## Check 5c - Additional DirectQuery optimizations

These are workload-dependent recommendations rather than universal pass/fail rules:

- **[Calendar-based time intelligence](https://blog.crossjoin.co.uk/2025/11/30/a-look-at-the-impact-of-calendar-based-time-intelligence-on-power-bi-directquery-performance/):** test it for YTD, prior-period, and similar calculations. It can calculate at a coarser grain and substantially reduce rows returned, CPU, memory, and duration. Benchmark with Performance Analyzer and Execution Metrics because it can also increase the number of SQL requests.
- **[Visual Calculations](https://learn.microsoft.com/power-bi/transform-model/desktop-visual-calculations-overview):** consider them for visual-local calculations such as running totals, moving averages, and comparisons. They operate on aggregated data already present in the visual and can avoid extra source-query work. They are report-level objects, so this model-only notebook cannot detect them.
- **[Hybrid Tables with `dataCoverageDefinition`](https://blog.crossjoin.co.uk/2024/02/25/datacoveragedefinition-a-new-optimisation-for-hybrid-tables-in-power-bi/):** an advanced option when historical data is stable but recent data must remain current. Define DirectQuery partition coverage so Power BI can skip the DirectQuery source when Import partitions fully answer a query.
- **[Discourage Implicit Measures](https://blog.crossjoin.co.uk/2023/04/16/disabling-filter-pane-aggregates-in-power-bi/):** set `Model.DiscourageImplicitMeasures = true` to encourage explicit measures and potentially suppress filter-pane count queries. Treat the query-reduction benefit as a compatibility check because current product behavior may vary.

## Check 6 — Query parallelism / MaxParallelismPerQuery
For F64+ capacities, raising `Model.MaxParallelismPerQuery` can improve DQ performance. Compare against the SKU table below.

In [ ]:
sku_limits = pd.DataFrame([
    ("F2",    3,   5,  "1"),
    ("F4",    3,   5,  "1"),
    ("F8",    3,  10,  "1"),
    ("F16",   5,  10,  "1"),
    ("F32",  10,  10,  "1"),
    ("F64",  25,  50,  "4-8"),
    ("F128", 50,  75,  "6-12"),
    ("F256", 100, 100, "8-16"),
    ("F512", 200, 200, "10-20"),
    ("F1024",400, 200, "12-24"),
    ("F2048",400, 200, "12-24"),
], columns=["SKU", "MaxMemoryGB", "MaxConcurrentDQConnections", "MaxDQParallelism"])
sku_limits

## Check 7 — Per-data-source `MaxConnections` (model-level DQ connection cap)

`MaxConcurrentDQConnections` (per-SKU) is a *capacity* limit and can't be changed. But each model data source has a tunable **`MaxConnections`** property (TOM: `DataSource.MaxConnections`, default `10`) that limits concurrent DQ connections opened by the engine against that source. It should be ≤ the SKU's `MaxConcurrentDQConnections` to avoid queueing.

This cell reads `MaxConnections` for every data source in the model via TOM and compares against an optional target SKU.

In [ ]:
# Target SKU of the capacity that hosts the workspace (used only for the comparison below).
TARGET_SKU = "F64"   # <- change to match your capacity: F2, F4, F8, F16, F32, F64, F128, F256, F512, F1024, F2048

ds_rows = []
try:
    # Reuse `model` from Check 5 if present, otherwise re-open.
    if "model" not in dir() or model is None:
        tom = fabric.create_tom_server(workspace=WORKSPACE)
        db = (tom.Databases.GetByName(DATASET)
              if hasattr(tom.Databases, "GetByName") else tom.Databases[DATASET])
        model = db.Model

    # 1) Classic (legacy) DataSource objects
    for ds in getattr(model, "DataSources", []) or []:
        ds_rows.append({
            "DataSource":     ds.Name,
            "Type":            str(getattr(ds, "Type", "Provider")),
            "MaxConnections":  getattr(ds, "MaxConnections", None),
            "ConnectionDetails": str(getattr(ds, "ConnectionString", "") or getattr(ds, "Account", "")),
        })

    # 2) Modern (Power Query / structured) sources live in partition M expressions.
    #    They don't expose MaxConnections individually; the model-level default (10) applies
    #    unless overridden via TOM scripting.
    pq_partitions = []
    for t in model.Tables:
        for p in t.Partitions:
            src = getattr(p, "Source", None)
            src_type = type(src).__name__ if src is not None else ""
            if "MPartitionSource" in src_type or "M" == src_type or "QueryPartitionSource" in src_type:
                pq_partitions.append((t.Name, p.Name, src_type))

    if not ds_rows and pq_partitions:
        print("Model uses Power Query / structured sources — no per-datasource "
              "`MaxConnections` object is exposed.")
        print(f"Power Query partitions: {len(pq_partitions)}")
        print("Engine default MaxConnections = 10 per source unless overridden by TOM scripting.")

except Exception as ex:
    print("TOM read failed:", ex)

ds_df = pd.DataFrame(ds_rows)
display(ds_df if not ds_df.empty else pd.DataFrame({"info": ["no legacy DataSource objects found"]}))

# Compare against the SKU's MaxConcurrentDQConnections
sku_row = sku_limits.loc[sku_limits["SKU"] == TARGET_SKU]
if not sku_row.empty and not ds_df.empty and "MaxConnections" in ds_df:
    sku_cap = int(sku_row.iloc[0]["MaxConcurrentDQConnections"])
    ds_df["SKU_Cap"] = sku_cap
    ds_df["OK"]      = ds_df["MaxConnections"].fillna(10).astype(int) <= sku_cap
    print(f"\nSKU {TARGET_SKU} cap = {sku_cap} concurrent DQ connections per semantic model")
    display(ds_df[["DataSource", "MaxConnections", "SKU_Cap", "OK"]])
    max_connections_ok = bool(ds_df["OK"].all())
else:
    max_connections_ok = None
    print("Could not evaluate MaxConnections vs SKU cap.")

## 6. Consolidated report

In [ ]:
report = []

report.append(("Model uses DirectQuery",
               "INFO", f"{is_dq_model}"))

report.append(("Star schema - snowflaked relationships",
               "PASS" if snowflake_rels.empty else "WARN",
               f"{len(snowflake_rels)} found"))

report.append(("Star schema - standalone tables",
               "PASS" if not standalone else "WARN",
               ", ".join(standalone) or "none"))

report.append(("Power Query transforms on DQ fact tables (create source view)",
               pq_transform_status if 'pq_transform_status' in dir() else "MANUAL",
               pq_transform_detail if 'pq_transform_detail' in dir() else "Run Check 1b to evaluate DQ M transformations."))

report.append(("Dimensions in Dual mode",
               "PASS" if not_dual.empty else "FAIL",
               f"{len(not_dual)} dimension(s) not Dual"))

report.append(("Assume Referential Integrity on all relationships",
               "PASS" if missing_ri.empty else "FAIL",
               f"{len(missing_ri)} relationship(s) missing RI"))

report.append(("Relationship keys are Integer",
               "PASS" if bad_types.empty else "FAIL",
               f"{len(bad_types)} relationship(s) use non-integer keys"))

auto_agg_detail = (
    f"Enabled (tom={auto_agg_tom}, service={auto_agg_service}); validate query benefit against CU consumption."
    if auto_agg_enabled
    else "Not enabled. This is acceptable; prefer workload-driven User-Defined Aggregations when needed."
)
report.append(("Auto Aggregations (informational, not a default recommendation)",
               "INFO", auto_agg_detail))

report.append(("SSO vs Auto Aggregations (use User-Defined Aggregations if SSO)",
               sso_status if 'sso_status' in dir() else "MANUAL",
               sso_detail if 'sso_detail' in dir() else "Run Check 5b to evaluate SSO on the data source."))

report.append(("Calendar-based time intelligence and Visual Calculations",
               "MANUAL",
               "Evaluate with Performance Analyzer and Execution Metrics for time-intelligence and visual-local calculations."))

hybrid_detail = (
    f"Hybrid tables: {', '.join(hybrid_tables)}; dataCoverageDefinition: {', '.join(data_coverage_tables) or 'none detected'}."
    if hybrid_tables
    else "No hybrid tables detected. Consider only for a suitable hot/cold data pattern."
)
report.append(("Hybrid Tables / dataCoverageDefinition (advanced)",
               "INFO", hybrid_detail))

if discourage_implicit_measures is True:
    discourage_status = "PASS"
    discourage_detail = "Model.DiscourageImplicitMeasures is enabled."
elif discourage_implicit_measures is False:
    discourage_status = "WARN"
    discourage_detail = ("Model.DiscourageImplicitMeasures is disabled. Consider enabling it after testing "
                          "report-authoring and filter-pane behavior.")
else:
    discourage_status = "MANUAL"
    discourage_detail = "Property could not be read; verify Model.DiscourageImplicitMeasures manually."
report.append(("Discourage Implicit Measures",
               discourage_status, discourage_detail))

report.append(("MaxParallelismPerQuery configured",
               "INFO", f"{max_parallel}"))

report.append(("DataSource MaxConnections <= SKU cap",
               "PASS" if max_connections_ok else ("WARN" if max_connections_ok is None else "FAIL"),
               f"SKU={TARGET_SKU}, sources={len(ds_df) if 'ds_df' in dir() else 0}"))

report_df = pd.DataFrame(report, columns=["Check", "Status", "Detail"])

# Sort by status order: PASS, INFO, WARN, MANUAL, FAIL.
STATUS_ORDER = ["PASS", "INFO", "WARN", "MANUAL", "FAIL"]
report_df["Status"] = pd.Categorical(report_df["Status"], categories=STATUS_ORDER, ordered=True)
report_df = report_df.sort_values("Status").reset_index(drop=True)

# Background + text colour per status.
STATUS_STYLE = {
    "PASS":   "background-color: #d4edda; color: #155724;",   # green
    "FAIL":   "background-color: #f8d7da; color: #721c24;",   # red
    "WARN":   "background-color: #fff3cd; color: #856404;",   # amber
    "MANUAL": "background-color: #cce5ff; color: #004085;",   # blue
    "INFO":   "background-color: #e2e3e5; color: #383d41;",   # grey
}

def _style_row(row):
    style = STATUS_STYLE.get(row["Status"], "")
    # Bold the Status cell; colour the whole row.
    return [style + (" font-weight: bold;" if col == "Status" else "") for col in row.index]

styled = (report_df.style
          .apply(_style_row, axis=1)
          .set_properties(**{"text-align": "left"})
          .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
          .hide(axis="index"))

styled

## 7. Remediation hints

- **Power Query transforms on DQ fact tables**: recreate the logic as a **view at the data source** and repoint the DirectQuery fact table at the view (keeps query folding clean and pushes work to the source).
- **Dimensions not Dual**: open the model in Power BI Desktop -> select the table -> *Advanced properties* -> **Storage mode = Dual**.
- **Assume RI missing**: edit the relationship -> check **Assume referential integrity**.
- **Non-integer keys**: replace string/GUID keys with surrogate `Int64` keys in Power Query or the source.
- **Apply slicer button**: File -> Options -> Query reduction -> *Add an Apply button to each slicer*.
- **Aggregations**: build targeted [User-Defined Aggregations](https://learn.microsoft.com/power-bi/transform-model/aggregations-advanced) from known report workloads. If Auto Aggregations are enabled, retain them only when measured query gains justify their CU consumption.
- **SSO source**: Auto Aggregations skip SSO-enabled DirectQuery tables; use User-Defined Aggregations that preserve the required security behavior.
- **Calendar-based time intelligence**: benchmark calendar-based YTD and prior-period calculations; compare duration, DirectQuery rows, SQL request count, CPU, and memory.
- **Visual Calculations**: use them for suitable visual-local calculations over aggregated results, especially running totals, moving averages, and period comparisons.
- **Hybrid Tables**: for hot/cold data patterns, consider Import plus DirectQuery partitions and define `dataCoverageDefinition` on the DirectQuery partition.
- **Discourage Implicit Measures**: test and set `Model.DiscourageImplicitMeasures = true` to favor explicit measures and potentially suppress filter-pane aggregate queries.
- **Parallelism**: use TOM scripting to set `Model.MaxParallelismPerQuery` (see the SKU table).

## 8. Further reading

**DirectQuery & modeling**
- [DirectQuery model guidance in Power BI Desktop](https://learn.microsoft.com/power-bi/guidance/directquery-model-guidance)
- [Use DirectQuery in Power BI Desktop](https://learn.microsoft.com/power-bi/connect-data/desktop-use-directquery)
- [Understand star schema and the importance for Power BI](https://learn.microsoft.com/power-bi/guidance/star-schema)

**Storage mode & relationships**
- [Manage storage mode in Power BI Desktop (Import / DirectQuery / Dual)](https://learn.microsoft.com/power-bi/transform-model/desktop-storage-mode)
- [Assume referential integrity settings in Power BI Desktop](https://learn.microsoft.com/power-bi/connect-data/desktop-assume-referential-integrity)
- [Model relationships - data types of relationship columns](https://learn.microsoft.com/power-bi/transform-model/desktop-relationships-understand#data-types-of-columns)

**Aggregations**
- [User-defined aggregations](https://learn.microsoft.com/power-bi/transform-model/aggregations-advanced)
- [Automatic aggregations](https://learn.microsoft.com/fabric/enterprise/powerbi/aggregations-auto) and [configuration](https://learn.microsoft.com/power-bi/enterprise/aggregations-auto-configure) (reference only; validate CU cost before recommending)

**Single sign-on (SSO)**
- [Overview of single sign-on for on-premises data gateways](https://learn.microsoft.com/power-bi/connect-data/service-gateway-sso-overview)
- [SSO considerations for the Power BI service](https://learn.microsoft.com/power-bi/connect-data/service-get-data#considerations-and-limitations)

**Advanced DirectQuery optimization**
- [Calendar-based time intelligence impact on DirectQuery performance](https://blog.crossjoin.co.uk/2025/11/30/a-look-at-the-impact-of-calendar-based-time-intelligence-on-power-bi-directquery-performance/)
- [Visual Calculations overview](https://learn.microsoft.com/power-bi/transform-model/desktop-visual-calculations-overview)
- [`dataCoverageDefinition` optimization for Hybrid Tables](https://blog.crossjoin.co.uk/2024/02/25/datacoveragedefinition-a-new-optimisation-for-hybrid-tables-in-power-bi/)
- [Table partitions and data coverage definitions](https://learn.microsoft.com/analysis-services/tom/table-partitions)
- [Disabling filter-pane aggregates with Discourage Implicit Measures](https://blog.crossjoin.co.uk/2023/04/16/disabling-filter-pane-aggregates-in-power-bi/)
- [`Model.DiscourageImplicitMeasures` (TOM API reference)](https://learn.microsoft.com/dotnet/api/microsoft.analysisservices.tabular.model.discourageimplicitmeasures)

**Query reduction & performance**
- [Optimize the ribbon - apply query reduction settings](https://learn.microsoft.com/power-bi/create-reports/desktop-optimize-ribbon-scenarios#apply-query-reduction-settings)
- [Evaluation configuration settings for Power BI Desktop](https://learn.microsoft.com/power-bi/create-reports/desktop-evaluation-configuration)
- [Query parallelization boosts DirectQuery performance (blog)](https://powerbi.microsoft.com/blog/query-parallelization-helps-to-boost-power-bi-dataset-performance-in-directquery-mode/)
- [`Model.MaxParallelismPerQuery` (TOM API reference)](https://learn.microsoft.com/dotnet/api/microsoft.analysisservices.tabular.model.maxparallelismperquery)